# Kapitel 5

## Faktafrågor

1. curse of dimensionality betyder att efter en viss punkt spelar nästan ingen roll om man har fler features. Fler featuers är inte alltid positivt, för att ju mer features vi får destå glesare blir datan och där med svårt att hitta meningsfulla mösnter. 

2. dimensionsreducering betyder att minskar antal featuers alltså dimensioner i datan. det gör att det blir mindre risk för överfitting och modellen blir snabbare. 

3. PCA är an analysmetod som hittar de riktningar i datan som förklarar mest varians och projicerar ner datan på dem. i figur 5.4 kan man se att man har projecerat 2d data på linjer 1d.

4. kernel PCA utvärderas oftast på två sätt, bevakad och ovebakad. 

om det är bevakad så kör man kernel pca och en klassificerare efteråt, sedan mätar man accuracy. man kan testa olika kernels och parametrar med gridsearchcv för att se vilken ger bäst resultat. 

om det är obevakad så mäter man hur mycket information som går fölorad. först krypmer man data och sedan återskapar den men inverse_transform och jämför med originalet, för att ta reda på reconstruction error. 

## Resonemangfrågor

5. Både har rätt för att man vill ha så bra prediktioner som möjligt men samtidigt vill man minimera modelltränings tiden så mycket som möjligt. 

6. Efter vi har genomfört PCA variablerna är inte längre skiljbara alla variabler kokas ihop och visas i form av en blanding via PCA.

## Koduppgifter

uppgift_8

In [14]:
import numpy as np
from sklearn.decomposition import PCA
# Creating a dataset with 3 features/columns
X = np.random.rand(1000, 3)
print(X[0:5])
# Reducing the data to 2 dimensions
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
print(X2D[0:5])
# "Recreating" the data to 3 dimensions
X3D_inv = pca.inverse_transform(X2D)
# Not exactly equal since some information was lost in the transformation
print(np.allclose(X3D_inv, X))

[[0.25112813 0.21894135 0.89925664]
 [0.14245075 0.22105431 0.53453441]
 [0.95957663 0.20223929 0.59817676]
 [0.46326159 0.83528051 0.19929404]
 [0.57719408 0.07949304 0.20351979]]
[[-0.40996279 -0.26817443]
 [-0.28163335 -0.35809127]
 [-0.32560903  0.4548912 ]
 [ 0.40879478 -0.02863237]
 [-0.30269313  0.09481335]]
False


koden ovan importerar PCA from sklearn.decomposition. 

sedan skapar data med 3 columner och 1000 rader. 

sedan instancierar man PCA där man minskar antal dimensioner från 3 till 2. 

sedan kör man en fit_transform på X med pca, som projicerar ner datan till x2d.

sedan försöker man återskapar datan från den 2D tillbaka till 3D med hjälp av inverse_tranform. 

np.allclose kollar om den orginal datan och den återskapade datan är nästa identiska. I det här fallet blir det falskt för att PCA tappar alltid data när man gör komprimering. 

uppgift_9

In [15]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

In [16]:
df = pd.read_csv("car_price_dataset.csv", sep=';')

In [17]:
df.head()

,Brand,Model,Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Price
0,Kia,Rio,2020,4.2,Diesel,Manual,289944,3,5,8501
1,Chevrolet,Malibu,2012,2.0,Hybrid,Automatic,5356,2,3,12092
2,Mercedes,GLA,2020,4.2,Diesel,Automatic,231440,4,2,11171
3,Audi,Q5,2023,2.0,Electric,Manual,160971,2,1,11780
4,Volkswagen,Golf,2003,2.6,Hybrid,Semi-Automatic,286618,3,3,2867


In [18]:
X = df.drop(columns=['Price'])
y = df['Price']

In [19]:
X_encoded = pd.get_dummies(X, columns=['Brand', 'Model', 'Fuel_Type', 'Transmission'])

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [21]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [22]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train)

In [23]:
linear_model  = LinearRegression()
linear_model.fit(X_train, y_train)

linear_model_pred = linear_model.predict(X_test)

In [24]:
linear_rmse = root_mean_squared_error( y_test, linear_model_pred)
print('rmse: ',linear_rmse)

rmse:  64.91473251756284


In [25]:
comparison = pd.DataFrame({
    'actual': y_test.values,
    'prediction': linear_model.predict(X_test)
})
comparison['error'] = comparison['actual'] - comparison['prediction']
print('sammanfatting av felmarginal:')
print(comparison['error'].describe())

sammanfatting av felmarginal:
count    2000.000000
mean       -2.236376
std        64.892424
min       -54.081831
25%       -19.885881
50%        -6.711651
75%         7.007655
max      1716.386449
Name: error, dtype: float64


slutsats: 

med pca fick jag rmse = 64.91

utan pca fick jag rmse  = 56.13 från kap 3

det vill säga att pca är inte fördelaktigt här för vi får sämmre rmse. och det är inget konstigt egentligen för att pca är bättre för datasets som har hundra eller tusen olika features, här har vi under 20 features. 

pca söker efter mest varians den bryr sig inte om y[price]. 